# BiDense Model Evaluation

In [1]:
import os
import torch
import numpy as np
from PIL import Image
import matplotlib.cm as cm
from tqdm.auto import tqdm

from configs.depth.default import get_cfg_defaults as get_depth_cfg
from configs.segmentation.default import get_cfg_defaults as get_seg_cfg
from datasets.depth_dataloader import get_dataLoader as get_depth_dataLoader
from datasets.segmentation_dataloader import get_dataLoader as get_seg_dataLoader
from pl_trainer_depth import PL_DepthTrainer
from pl_trainer_segmentation import PL_SegmentationTrainer
from misc import (
    inv_normalize, compute_depth_metrics,
    compute_segmentation_metrics, visualize_segmentation_result,
)

# ==============================================================
#  Parameters (modify as needed)
# ==============================================================
DEVICE = 'cuda:0'
OUTPUT_DIR = 'test_results'

# Sample indices to save as images (parameterizable)
NYU_SAMPLE_INDICES = [0, 10, 50, 100, 200]
ADE20K_SAMPLE_INDICES = [0, 10, 50, 100, 200]
VOC_SAMPLE_INDICES = [0, 10, 50, 100, 200]

# Best checkpoint paths
NYU_CKPT = 'lightning_logs/version_7/checkpoints/epoch=47-step=36384-v1.ckpt'
ADE20K_CKPT = 'lightning_logs/version_8/checkpoints/epoch=49-step=31600-v1.ckpt'
VOC_CKPT = 'lightning_logs/version_9/checkpoints/epoch=199-step=18200-v1.ckpt'

# TensorBoard log directories (original training + resumed)
NYU_LOG_DIRS = ['lightning_logs/version_4', 'lightning_logs/version_7']
ADE20K_LOG_DIRS = ['lightning_logs/version_5', 'lightning_logs/version_8']
VOC_LOG_DIRS = ['lightning_logs/version_6', 'lightning_logs/version_9']

# ==============================================================
#  Helper functions
# ==============================================================
def save_input_image(tensor, path):
    """Save a normalized image tensor as PNG."""
    img = inv_normalize(tensor).clamp(0, 1)
    img = (img.permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
    Image.fromarray(img).save(path)

def save_depth_image(depth_tensor, path):
    """Save depth tensor as colorized PNG (plasma colormap)."""
    d = depth_tensor.squeeze().cpu().numpy()
    d_norm = (d - d.min()) / (d.max() - d.min() + 1e-8)
    colored = (cm.plasma(d_norm)[:, :, :3] * 255).astype(np.uint8)
    Image.fromarray(colored).save(path)

def save_seg_image(mask_tensor, path, dataset):
    """Save segmentation mask as colorized PNG using dataset palette."""
    vis = visualize_segmentation_result(mask_tensor, dataset)
    img = (vis.permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
    Image.fromarray(img).save(path)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Setup complete.')

/home/ljs/anaconda3/envs/artwork/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Setup complete.


/home/ljs/anaconda3/envs/artwork/lib/python3.8/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [2]:
# ==============================================================
#  Training / Validation Metrics (from TensorBoard logs)
# ==============================================================
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

def load_training_metrics(log_dirs, tags=None):
    data = {}
    for d in log_dirs:
        ea = EventAccumulator(d)
        ea.Reload()
        for tag in ea.Tags().get('scalars', []):
            if tags and tag not in tags:
                continue
            if tag not in data:
                data[tag] = []
            data[tag].extend([(e.step, e.value) for e in ea.Scalars(tag)])
    for tag in data:
        data[tag].sort(key=lambda x: x[0])
    return data

def print_metrics(data, title):
    print(f"\n{'='*55}")
    print(f"  {title}")
    print(f"{'='*55}")
    for tag, vals in sorted(data.items()):
        if vals:
            print(f"  {tag:30s} = {vals[-1][1]:.4f}")

nyu_train = load_training_metrics(NYU_LOG_DIRS, [
    'train/loss', 'valid/loss', 'valid/d1', 'valid/d2', 'valid/d3',
    'valid/abs_rel', 'valid/rms', 'valid/log10', 'valid/silog',
    'valid/sq_rel', 'valid/log_rms',
])
print_metrics(nyu_train, 'NYU Depth - Final Training/Validation Metrics')

ade_train = load_training_metrics(ADE20K_LOG_DIRS, [
    'train/loss', 'valid/loss', 'valid/mIoU', 'valid/pixAcc',
])
print_metrics(ade_train, 'ADE20K - Final Training/Validation Metrics')

voc_train = load_training_metrics(VOC_LOG_DIRS, [
    'train/loss', 'valid/loss', 'valid/mIoU', 'valid/pixAcc',
])
print_metrics(voc_train, 'Pascal VOC - Final Training/Validation Metrics')


  NYU Depth - Final Training/Validation Metrics
  train/loss                     = 0.9215
  valid/abs_rel                  = 0.1923
  valid/d1                       = 0.7038
  valid/d2                       = 0.9217
  valid/d3                       = 0.9784
  valid/log10                    = 0.0791
  valid/log_rms                  = 0.2296
  valid/loss                     = 1.9486
  valid/rms                      = 0.6207
  valid/silog                    = 18.5051
  valid/sq_rel                   = 0.1659

  ADE20K - Final Training/Validation Metrics
  train/loss                     = 1.2572
  valid/loss                     = 1.3418
  valid/mIoU                     = 0.1551
  valid/pixAcc                   = 0.6397

  Pascal VOC - Final Training/Validation Metrics
  train/loss                     = 0.1174
  valid/loss                     = 0.2211
  valid/mIoU                     = 0.7518
  valid/pixAcc                   = 0.9256


## 1. NYU Depth Estimation

In [3]:
config = get_depth_cfg()
config.merge_from_file('configs/depth/upernet_bidense_nyu.yaml')

model = PL_DepthTrainer.load_from_checkpoint(NYU_CKPT, map_location=DEVICE)
model.eval()
model.to(DEVICE)

dl = get_depth_dataLoader(
    mode='online_eval', batch_size=1, num_threads=4,
    dataset=config.DATASET.DATASET,
    data_path=config.DATASET.DATA_PATH,
    gt_path=config.DATASET.GT_PATH,
    filenames_file=config.DATASET.FILENAMES_FILE,
    data_path_eval=config.DATASET.DATA_PATH_EVAL,
    gt_path_eval=config.DATASET.GT_PATH_EVAL,
    filenames_file_eval=config.DATASET.FILENAMES_FILE_EVAL,
    input_height=config.DATASET.INPUT_HEIGHT,
    input_width=config.DATASET.INPUT_WIDTH,
    do_random_rotate=False, degree=0,
    do_kb_crop=config.PREPROCESSING.DO_KB_CROP, use_right=False,
)

out_dir = os.path.join(OUTPUT_DIR, 'nyu_depth')
os.makedirs(out_dir, exist_ok=True)

all_metrics = []
with torch.no_grad():
    for idx, batch in enumerate(tqdm(dl, desc='NYU Depth')):
        image = batch['image'].to(DEVICE)
        depth_gt = batch['depth'].to(DEVICE)

        depth_est = model.module(image)

        metrics = compute_depth_metrics(
            depth_gt, depth_est,
            garg_crop=config.ONLINE_EVAL.GARG_CROP,
            eigen_crop=config.ONLINE_EVAL.EIGEN_CROP,
            dataset='nyu',
            min_depth_eval=config.ONLINE_EVAL.MIN_DEPTH_EVAL,
            max_depth_eval=config.ONLINE_EVAL.MAX_DEPTH_EVAL,
        )
        all_metrics.append([m.item() for m in metrics])

        if idx in NYU_SAMPLE_INDICES:
            save_input_image(image[0], os.path.join(out_dir, f'{idx:04d}_input.png'))
            save_depth_image(1.0 / depth_gt[0].clamp(min=1e-3), os.path.join(out_dir, f'{idx:04d}_depth_gt.png'))
            save_depth_image(1.0 / depth_est[0].clamp(min=1e-3), os.path.join(out_dir, f'{idx:04d}_depth_pred.png'))

all_metrics = np.array(all_metrics)
mean = all_metrics.mean(axis=0)
names = ['silog', 'abs_rel', 'log10', 'rms', 'sq_rel', 'log_rms', 'd1', 'd2', 'd3']

print(f"\n{'='*55}")
print(f"  NYU Depth - Test Metrics")
print(f"{'='*55}")
for n, v in zip(names, mean):
    print(f"  {n:15s} = {v:.4f}")
print(f"\nVisualizations saved to: {out_dir}/")

del model
torch.cuda.empty_cache()

/home/ljs/anaconda3/envs/artwork/lib/python3.8/site-packages/lightning/fabric/utilities/cloud_io.py:57: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
NYU Depth: 100%|██████████| 654/654 [00:


  NYU Depth - Test Metrics
  silog           = 18.4602
  abs_rel         = 0.1917
  log10           = 0.0790
  rms             = 0.6210
  sq_rel          = 0.1655
  log_rms         = 0.2291
  d1              = 0.7059
  d2              = 0.9220
  d3              = 0.9781

Visualizations saved to: test_results/nyu_depth/


## 2. ADE20K Semantic Segmentation

In [4]:
config = get_seg_cfg()
config.merge_from_file('configs/segmentation/upernet_bidense_ade20k.yaml')

model = PL_SegmentationTrainer.load_from_checkpoint(ADE20K_CKPT, map_location=DEVICE)
model.eval()
model.to(DEVICE)

dl = get_seg_dataLoader(
    dataset='ade20k', mode='val',
    data_path=config.DATASET.DATA_PATH,
    base_size=config.DATASET.BASE_SIZE,
    crop_size=config.DATASET.CROP_SIZE,
    batch_size=1, num_threads=4,
)

out_dir = os.path.join(OUTPUT_DIR, 'ade20k')
os.makedirs(out_dir, exist_ok=True)

n_classes = config.MODEL.NUM_CLASSES - 1
total_inter, total_union = 0, 0
all_pixacc = []

with torch.no_grad():
    for idx, batch in enumerate(tqdm(dl, desc='ADE20K')):
        image = batch['image'].to(DEVICE)
        mask_gt = batch['mask'].to(DEVICE)

        pred = model.module(image)

        pixAcc, area_inter, area_union = compute_segmentation_metrics(mask_gt, pred, n_classes)
        total_inter = total_inter + area_inter
        total_union = total_union + area_union
        all_pixacc.append(pixAcc.item())

        if idx in ADE20K_SAMPLE_INDICES:
            save_input_image(image[0], os.path.join(out_dir, f'{idx:04d}_input.png'))
            save_seg_image(mask_gt[0], os.path.join(out_dir, f'{idx:04d}_seg_gt.png'), 'ade20k')
            save_seg_image(pred[0].argmax(0), os.path.join(out_dir, f'{idx:04d}_seg_pred.png'), 'ade20k')

total_union_safe = total_union.clone().float()
total_union_safe[total_union_safe == 0] = torch.inf
mIoU = (total_inter.float() / total_union_safe).mean().item()

print(f"\n{'='*55}")
print(f"  ADE20K - Test Metrics")
print(f"{'='*55}")
print(f"  {'mIoU':15s} = {mIoU:.4f}")
print(f"  {'pixAcc':15s} = {np.mean(all_pixacc):.4f}")
print(f"\nVisualizations saved to: {out_dir}/")

del model
torch.cuda.empty_cache()

ADE20K: 100%|██████████| 2000/2000 [02:56<00:00, 11.33it/s]



  ADE20K - Test Metrics
  mIoU            = 0.1585
  pixAcc          = 0.6363

Visualizations saved to: test_results/ade20k/


## 3. Pascal VOC Semantic Segmentation

In [5]:
config = get_seg_cfg()
config.merge_from_file('configs/segmentation/upernet_bidense_pascal_voc.yaml')

model = PL_SegmentationTrainer.load_from_checkpoint(VOC_CKPT, map_location=DEVICE)
model.eval()
model.to(DEVICE)

dl = get_seg_dataLoader(
    dataset='pascal_voc', mode='val',
    data_path=config.DATASET.DATA_PATH,
    base_size=config.DATASET.BASE_SIZE,
    crop_size=config.DATASET.CROP_SIZE,
    batch_size=1, num_threads=4,
)

out_dir = os.path.join(OUTPUT_DIR, 'pascal_voc')
os.makedirs(out_dir, exist_ok=True)

n_classes = config.MODEL.NUM_CLASSES - 1
total_inter, total_union = 0, 0
all_pixacc = []

with torch.no_grad():
    for idx, batch in enumerate(tqdm(dl, desc='Pascal VOC')):
        image = batch['image'].to(DEVICE)
        mask_gt = batch['mask'].to(DEVICE)

        pred = model.module(image)

        pixAcc, area_inter, area_union = compute_segmentation_metrics(mask_gt, pred, n_classes)
        total_inter = total_inter + area_inter
        total_union = total_union + area_union
        all_pixacc.append(pixAcc.item())

        if idx in VOC_SAMPLE_INDICES:
            save_input_image(image[0], os.path.join(out_dir, f'{idx:04d}_input.png'))
            save_seg_image(mask_gt[0], os.path.join(out_dir, f'{idx:04d}_seg_gt.png'), 'pascal_voc')
            save_seg_image(pred[0].argmax(0), os.path.join(out_dir, f'{idx:04d}_seg_pred.png'), 'pascal_voc')

total_union_safe = total_union.clone().float()
total_union_safe[total_union_safe == 0] = torch.inf
mIoU = (total_inter.float() / total_union_safe).mean().item()

print(f"\n{'='*55}")
print(f"  Pascal VOC - Test Metrics")
print(f"{'='*55}")
print(f"  {'mIoU':15s} = {mIoU:.4f}")
print(f"  {'pixAcc':15s} = {np.mean(all_pixacc):.4f}")
print(f"\nVisualizations saved to: {out_dir}/")

del model
torch.cuda.empty_cache()

1449it [00:00, 121255.07it/s]
Pascal VOC: 100%|██████████| 1449/1449 [02:10<00:00, 11.08it/s]


  Pascal VOC - Test Metrics
  mIoU            = 0.7524
  pixAcc          = 0.9250

Visualizations saved to: test_results/pascal_voc/


## 4. Binary Kernel Weight Visualization

In [6]:
import torch
from pl_trainer_depth import PL_DepthTrainer
from pl_trainer_segmentation import PL_SegmentationTrainer

NYU_CKPT = 'lightning_logs/version_7/checkpoints/epoch=47-step=36384-v1.ckpt'
ADE20K_CKPT = 'lightning_logs/version_8/checkpoints/epoch=49-step=31600-v1.ckpt'
VOC_CKPT = 'lightning_logs/version_9/checkpoints/epoch=199-step=18200-v1.ckpt'

models = {
    'nyu': PL_DepthTrainer.load_from_checkpoint(NYU_CKPT, map_location='cpu'),
    'ade20k': PL_SegmentationTrainer.load_from_checkpoint(ADE20K_CKPT, map_location='cpu'),
    'pascal_voc': PL_SegmentationTrainer.load_from_checkpoint(VOC_CKPT, map_location='cpu'),
}
print('3 models loaded.')

3 models loaded.


In [10]:
import os
import torch.nn.functional as F
from PIL import Image
from torchvision import transforms

# 이미지 경로 파라미터
IMAGE_PATHS = {
    'nyu':        'test_results/nyu_depth/0000_input.png',
    'ade20k':     'test_results/ade20k/0000_input.png',
    'pascal_voc': 'test_results/pascal_voc/0000_input.png',
}

# (출력 채널, 입력 채널)
KERNEL_INDICES = {
    'nyu':        (0, 1),
    'ade20k':     (0, 1),
    'pascal_voc': (0, 1),
}

# feature map 내 공간 위치 (H, W)
H_POS = 5
W_POS = 5

normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

for name, (out_ch, in_ch) in KERNEL_INDICES.items():
    model = models[name]
    model.eval()

    # --- 가중치 ---
    weight = model.module.out_conv1.conv.weight.data
    kernel = weight[out_ch, in_ch]
    binary_kernel = torch.sign(kernel)
    scaling_factor = weight[out_ch].abs().mean()

    print(f"{'='*55}")
    print(f"  {name} | out_conv1.conv | shape: {list(weight.shape)}")
    print(f"  Kernel [out_ch={out_ch}, in_ch={in_ch}]")
    print(f"  Binary weights:\n{binary_kernel}")
    print(f"  Scaling factor: {scaling_factor:.6f}")

    # --- feature map 추출 ---
    img_path = IMAGE_PATHS[name]
    img = Image.open(img_path).convert('RGB')
    img_tensor = normalize(transforms.ToTensor()(img)).unsqueeze(0)

    captured = {}
    def make_hook(storage):
        def hook_fn(module, inp, out):
            storage['input'] = inp[0].detach().cpu()
            storage['output'] = out.detach().cpu()
        return hook_fn

    handle = model.module.out_conv1.register_forward_hook(make_hook(captured))
    with torch.no_grad():
        model.module(img_tensor)
    handle.remove()

    # --- feature map 저장 ---
    out_dir = os.path.dirname(img_path)
    basename = os.path.splitext(os.path.basename(img_path))[0].replace('_input', '')

    input_fm_path = os.path.join(out_dir, f'{basename}_input_feature_map.pt')
    output_fm_path = os.path.join(out_dir, f'{basename}_output_feature_map.pt')
    torch.save(captured['input'], input_fm_path)
    torch.save(captured['output'], output_fm_path)

    print(f"\n  Input  feature map: {list(captured['input'].shape)} → {input_fm_path}")
    print(f"  Output feature map: {list(captured['output'].shape)} → {output_fm_path}")

    # --- 3x3 패치 추출 및 MAC 연산 ---
    fm = captured['input']
    _, _, fH, fW = fm.shape
    fm_padded = F.pad(fm, (1, 1, 1, 1), mode='constant', value=0)
    patch = fm_padded[0, in_ch, H_POS:H_POS+3, W_POS:W_POS+3]

    elementwise = binary_kernel * scaling_factor * patch
    mac = elementwise.sum().item()

    print(f"\n  Input patch [in_ch={in_ch}, h={H_POS}, w={W_POS}] (3x3):")
    print(f"{patch}")
    print(f"\n  Element-wise (binary_kernel * scale * patch):")
    print(f"{elementwise}")
    print(f"\n  MAC = {mac:.6f}  (feature map size: {fH}x{fW})")
    print()

  nyu | out_conv1.conv | shape: [128, 96, 3, 3]
  Kernel [out_ch=0, in_ch=1]
  Binary weights:
tensor([[ 1.,  1.,  1.],
        [-1., -1., -1.],
        [ 1., -1.,  1.]])
  Scaling factor: 0.004132

  Input  feature map: [1, 96, 120, 160] → test_results/nyu_depth/0000_input_feature_map.pt
  Output feature map: [1, 128, 120, 160] → test_results/nyu_depth/0000_output_feature_map.pt

  Input patch [in_ch=1, h=5, w=5] (3x3):
tensor([[-0.2410, -0.2666, -0.1712],
        [-0.1536, -0.0674, -0.0160],
        [ 0.3894,  0.7119,  0.7567]])

  Element-wise (binary_kernel * scale * patch):
tensor([[-9.9572e-04, -1.1017e-03, -7.0749e-04],
        [ 6.3490e-04,  2.7852e-04,  6.5918e-05],
        [ 1.6089e-03, -2.9416e-03,  3.1270e-03]])

  MAC = -0.000031  (feature map size: 120x160)

  ade20k | out_conv1.conv | shape: [128, 96, 3, 3]
  Kernel [out_ch=0, in_ch=1]
  Binary weights:
tensor([[-1., -1., -1.],
        [-1.,  1., -1.],
        [ 1.,  1.,  1.]])
  Scaling factor: 0.009503

  Input  featur